# 02 — Factor Research

Compute the **Alpha158** feature set (qlib, from scratch) plus classical factors, then evaluate predictive power with an **alphalens-style** IC / ICIR / turnover / quantile-returns report and a factor-decay analysis.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM", "PG", "KO"]
prices = load_prices(symbols)
returns = prices.pct_change().dropna()
print("universe:", list(prices.columns), prices.shape)

universe: ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'JPM', 'XOM', 'PG', 'KO'] (2118, 8)


## Alpha158 features (single symbol)

In [3]:
from alpha.feature_engineering.alpha158 import Alpha158

ohlcv = synth_ohlcv(prices[symbols[0]])
feats = Alpha158().compute(ohlcv)
print("Alpha158 produced", feats.shape[1], "features")
assert feats.shape[1] == 158
feats.dropna().iloc[-1].head(12)

Alpha158 produced 158 features


KMID    -0.012499
KLEN     0.002793
KMID2   -4.474864
KUP     -0.012074
KUP2    -4.322790
KLOW     0.002368
KLOW2    0.847926
KSFT     0.001944
KSFT2    0.695853
OPEN0    1.012657
HIGH0    1.000430
LOW0     0.997602
Name: 2026-06-05 00:00:00, dtype: float64

## Classical cross-sectional factors

In [4]:
from alpha.factors.classical.momentum import MomentumFactor
from alpha.factors.classical.low_vol import LowVolFactor

mom = MomentumFactor(lookback=126, gap=21)
mom_panel = mom.compute(prices)
mom_z = mom.cross_sectional_zscore(mom_panel)
lowvol_panel = LowVolFactor(window=63).compute(prices)
print("momentum panel:", mom_panel.shape, "| non-null rows:", int(mom_panel.notna().any(axis=1).sum()))
mom_z.dropna(how="all").tail(3)

momentum panel: (2118, 8) | non-null rows: 1992


,AAPL,MSFT,AMZN,NVDA,JPM,XOM,PG,KO
date,,,,,,,,
2026-06-03,-0.636064,-1.586685,0.743918,0.295547,-0.436039,1.966035,-0.574332,0.227619
2026-06-04,-0.560668,-1.652667,0.678742,0.077576,-0.412166,1.999496,-0.459667,0.329355
2026-06-05,-0.591674,-1.814260,0.853878,0.633651,-0.542187,1.593415,-0.505732,0.372909


## Alphalens report (IC, ICIR, turnover, quantiles)

In [5]:
from alpha.validation.alphalens_report import AlphalensReport

fwd_returns = prices.pct_change(5).shift(-5)   # 5-day forward return
factor = mom_z.dropna(how="all")
report = AlphalensReport(factor, fwd_returns, quantiles=4).compute()
turn = float(np.asarray(report["turnover"]).mean())
ls = float(np.asarray(report["long_short_return"]).mean())
print("IC mean  : %+.4f" % report["ic_mean"])
print("ICIR     : %+.4f" % report["icir"])
print("IC t-stat: %+.3f" % report["ic_tstat"])
print("turnover : %.3f" % turn)
print("long-short: %+.5f" % ls)

IC mean  : +0.0247
ICIR     : +0.0509
IC t-stat: +2.267
turnover : nan
long-short: +0.00230


## Factor decay

In [6]:
from alpha.validation.factor_decay import FactorDecay

decay = FactorDecay().compute(factor, returns, max_lag=10)
print(decay.round(4).to_string())

     ic_mean  ic_std    icir
lag                         
0     0.0218  0.4824  0.0452
1     0.0218  0.4841  0.0451
2     0.0310  0.4803  0.0646
3     0.0326  0.4876  0.0668
4     0.0271  0.4880  0.0556
5     0.0247  0.4866  0.0509
6     0.0240  0.4821  0.0498
7     0.0278  0.4796  0.0580
8     0.0257  0.4805  0.0535
9     0.0277  0.4824  0.0573
10    0.0244  0.4802  0.0508


In [7]:
ic = report["ic"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic.cumsum().plot(ax=ax[0]); ax[0].set_title("Cumulative IC"); ax[0].axhline(0, color="k", lw=0.5)
decay["ic_mean"].plot(kind="bar", ax=ax[1]); ax[1].set_title("Mean IC by forward lag")
plt.tight_layout(); print("factor research complete.")

factor research complete.
